# Week 1 · 数据 + Tokenizer

> **本周一句话**:把 chinese-poetry 仓库的 5 万首唐诗,变成 train.pt / val.pt / tokenizer.pkl 三个文件,后面所有 Week 都在这三个文件之上。

没有模型,没有训练。但**数据 pipeline 跑通是后面 7 周的基石** —— 任何 bug 在这里没修干净,后面所有 loss 都不可信。

## 0. 本周目标

学完之后你应该能:
- 解释一份原始唐诗 JSON 是怎么变成模型能吃的 `torch.Tensor` 的
- 知道字符级 tokenizer vs BPE 各自的取舍
- 看懂 `get_batch` 里那个 `torch.randint(len(d) - block_size - 1)` 为什么是这个数字
- 一条命令 `python data/prepare.py` 复现整套数据

**产物**:
| 文件 | 大小 | 含义 |
|---|---|---|
| `data/processed/tang_poems_clean.txt` | 11 MB | 清洗后的纯文本 |
| `data/processed/tokenizer.pkl` | 0.2 MB | `stoi` / `itos` 字典 |
| `data/processed/train.pt` | 32 MB | 编码后的训练张量(int64) |
| `data/processed/val.pt` | 4 MB | 编码后的验证张量 |

## 1. 前置知识

**必备**:
- Python list/dict、文件 IO
- `torch.Tensor` 基本操作:`shape`、`dtype`、索引、`torch.stack`
- 知道 GPU 显存的概念

**不必备(看不懂可以跳过)**:
- BPE / SentencePiece 算法细节
- HuggingFace tokenizers 库 —— 我们故意不用,从零写最简单的字符级版本

## 2. 核心概念

### 2.1 数据 pipeline 五步走

```
  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐
  │  下载       │ → │  合并        │ → │  清洗        │
  │ git clone   │    │ JSON→txt    │    │  去噪音字符 │
  └─────────────┘    └─────────────┘    └─────────────┘
         ↓                                       ↓
   raw/chinese-poetry/             processed/tang_poems_clean.txt

         ┌─────────────┐    ┌─────────────────────────────┐
         │  建词表     │ → │  编码 + 切分                  │
         │  stoi/itos  │    │  text→tensor→train/val      │
         └─────────────┘    └─────────────────────────────┘
              ↓                       ↓
       tokenizer.pkl         train.pt + val.pt
```

每一步都对应一个独立的 `.py` 文件,可以单独跑也可以用 `data/prepare.py` 一键串起来。
**为什么拆这么细**:任何一步出问题,只需要重跑那一步 —— 数据 pipeline 是项目里最容易引入隐藏 bug 的环节,组合的边界要清楚。

### 2.2 为什么字符级 tokenizer

字符级 = 一个汉字一个 token。统计后 `vocab_size = 9563`(唐诗里所有不重复的字符 + 标点 + 换行)。

**为什么不用 BPE**?

| 维度 | 字符级 | BPE |
|---|---|---|
| 实现复杂度 | 5 行 Python | 需要训练合并规则,几百行 |
| 中文友好度 | ★★★(一字一意,天然 token) | ★★(子词概念在中文里不自然) |
| 词表大小 | 9k(唐诗)~ 14k(唐宋全)| 通常 32k+ |
| 学习成本 | 看 30 秒 | 看半天 |

**结论**:Week 1-7 用字符级,把概念跑通;真要工业级再换 BPE 不晚。

In [ ]:
# 5 行字符级 tokenizer 的完整实现
text = "床前明月光，疑是地上霜"
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

print(f"vocab_size = {len(chars)}")
print(f"encoded = {[stoi[c] for c in text]}")
print(f"round-trip = {''.join(itos[i] for i in [stoi[c] for c in text])}")

### 2.3 train/val 切分

我们简单按时间顺序切 90/10:前 90% 训练,后 10% 验证。

**为什么不随机切**?语言数据有强时序结构 —— 同一首诗的前后两句不能一句在 train、一句在 val,否则验证集 "泄露"。按位置切是最简单干净的隔离。

**val 拿来干什么**?每隔几百步算一次 `estimate_loss(val)`,看 val loss 走势:
- val 跟 train 一起降 → 在学规律
- val 卡住、train 还在降 → **开始过拟合**
- val 反弹 → 大概率过拟合 / lr 太大 / 数据有问题

val loss 是后面 7 周里最常看的数字,做个心理准备。

### 2.4 get_batch 的滑动窗口采样

这是 Week 1 里**最容易讲不清楚**的一个函数。完整代码:

```python
def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i : i + block_size]     for i in ix])
    y = torch.stack([d[i + 1 : i + 1 + block_size] for i in ix])
    return x, y
```

**关键三句**:

1. `torch.randint(len(d) - block_size - 1, (batch_size,))`
   抽 `batch_size` 个随机起点,每个起点要保证后面还有 `block_size + 1` 个字符可取 —— 所以最大起点是 `len(d) - block_size - 1`。

2. `x = d[i : i + block_size]`
   从起点取连续 128 个字作为输入。

3. `y = d[i + 1 : i + 1 + block_size]`
   错位一个字作为目标 —— 这就是"下一字预测"的本质:在每个位置,模型看 x[0:t+1],预测 y[t] = x[t+1]。

**为什么是 -block_size - 1 而不是 -block_size**?因为 y 比 x 还要多看一个未来字,边界少一个。这里出错最常见的表现是训练偶尔崩在 "IndexError: index out of bounds"。

In [ ]:
# 演示 get_batch 的输出形状
import torch

data = torch.arange(100, 200)   # 假装是 100 个 token 的连续数据
block_size, batch_size = 8, 3

ix = torch.randint(len(data) - block_size - 1, (batch_size,))
x = torch.stack([data[i : i + block_size]     for i in ix])
y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])

print(f"起点: {ix.tolist()}")
print(f"x.shape = {x.shape}  (batch_size, block_size)")
print(f"y.shape = {y.shape}")
print(f"x[0] = {x[0].tolist()}")
print(f"y[0] = {y[0].tolist()}   ← 注意 y[0][t] == x[0][t+1]")
print(f"对齐验证: {torch.equal(x[0, 1:], y[0, :-1])}")

## 3. 代码地图

按"读代码的顺序"列,不是字母序:

| 文件 | 行数 | 干什么 |
|---|---|---|
| `configs/config.py` | 80 | 所有路径常量,先看一眼 |
| `data/download.py` | 25 | `git clone` chinese-poetry |
| `data/merge.py` | 35 | 把所有 JSON 里的 `paragraphs` 拼成一个大 txt |
| `data/clean.py` | 25 | 去掉数字 / 几何符号 / 多余换行 |
| `tokenizer/tokenizer.py` | 40 | `CharTokenizer` 类,带 `save/load` |
| `tokenizer/build.py` | 35 | 读清洗后文本 → 建 tokenizer → 编码 → 切分 → 存 |
| `data/prepare.py` | 30 | 一键跑完上面 4 步 |
| `train/utils.py:get_batch` | 15 | 上面 2.4 节那个滑动窗口 |

`prepare.py` 是入口;`utils.py:get_batch` 是后面所有训练循环都会用的工具,重点看。

## 4. 动手做

**先激活 venv**(假设你已经 `uv sync` 过):

```powershell
.venv\Scripts\Activate.ps1
```

然后:

In [ ]:
# 一键跑完整 pipeline
# 注意: 必须用项目 .venv 里的 python 跑, 不能写死 "python"。
# 否则当前 Jupyter kernel 若是 anaconda / 系统 python(没装 torch),
# 会在 [4/4] import torch 时报 ModuleNotFoundError -> 退出码 1。
#
# 用 Popen 逐行读子进程输出, 让 prepare.py 的 [1/4]..[4/4] 进度实时显示在 cell 里。
# (subprocess.run 默认把子进程 stdout 写到 kernel 原始 fd, notebook 看不到。)
import subprocess, sys
from pathlib import Path


def _find_repo_root() -> Path:
    """定位仓库根目录(含 pyproject.toml + train/),不依赖 notebook 工作目录。
    本地/Colab 的 cwd 在 courses/,往上一级命中;
    魔搭 ModelScope 的 cwd 在工作区根(= 仓库根),当场命中。"""
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "pyproject.toml").is_file() and (d / "train").is_dir():
            return d
    # 兜底:往下找两层(仓库被克隆进子目录的情况)
    for f in (*here.glob("*/pyproject.toml"), *here.glob("*/*/pyproject.toml")):
        if (f.parent / "train").is_dir():
            return f.parent.resolve()
    raise RuntimeError(f"找不到仓库根(应含 pyproject.toml + train/),当前 cwd={here}")


REPO = _find_repo_root()                      # 项目根的绝对路径(后续 cell 也会用到)
print(f"REPO = {REPO}")
venv_py = REPO / ".venv" / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
py = str(venv_py) if venv_py.exists() else sys.executable   # 优先 .venv, 否则退回当前 kernel

print(f"用解释器: {py}")

proc = subprocess.Popen(
    [py, str(REPO / "data" / "prepare.py")],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,   # 合并 stderr 到 stdout
    text=True, encoding="utf-8", bufsize=1,             # encoding 必填(否则 GBK 解码生僻字会崩); bufsize=1 = 行缓冲
    cwd=str(REPO),
)
for line in proc.stdout:        # 子进程每打印一行, 这里立刻收到并显示
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"prepare.py 退出码 {proc.returncode}")

**预期输出**(关键行):

```
[1/4] 下载数据
  clone https://github.com/chinese-poetry/...

[2/4] 合并 JSON
  准备处理 58 个文件
  共 57603 首诗,3,993,966 字符

[3/4] 清洗噪音
  清洗前: 3,993,966 字符, vocab=9583
  清洗后: 3,987,419 字符, vocab=9563

[4/4] 构建 tokenizer + 编码切分
  vocab_size = 9563
  train: torch.Size([3588677])
  val:   torch.Size([398742])
```

**怎么判断跑对了**:
- `vocab_size` 是 **9563**(±2,版本差异)
- `train` 张量约 360 万 token,`val` 约 40 万
- 比例约 9:1
- 总大小 `train.pt + val.pt` 约 64 MB

任何一个数字差一个数量级就要看 log,别往下走。

In [ ]:
# 验证产物:加载 tokenizer 编码两个字看看
import sys
sys.path.insert(0, str(REPO))                 # REPO 在上面跑 prepare 的 cell 已定位

from tokenizer.tokenizer import CharTokenizer
from configs.config import TOKENIZER_FILE

tok = CharTokenizer.load(TOKENIZER_FILE)
print(f"vocab_size = {tok.vocab_size}")
print(f"'月' 的 ID = {tok.stoi['月']}")
print(f"'床前明月光' 编码 = {tok.encode('床前明月光')}")
print(f"还原 = {tok.decode(tok.encode('床前明月光'))}")

## 5. 自测题(不给答案,自己脑子里答)

**A. 数据**
- A1 `chinese-poetry` 仓库的目录结构是什么?`paragraphs` 字段长什么样?
- A2 清洗为什么把 `0123456789` 这种数字字符也去掉?去之前 vocab 是多少,去之后是多少?
- A3 如果不调 `re.sub(r"\n{3,}", "\n\n", text)`,会发生什么?对训练有什么影响?

**B. Tokenizer**
- B1 `stoi` 和 `itos` 是不是一定要从同一份数据构建?如果训练用的 `stoi` 和推理用的 `itos` 来自不同文本,会怎样?
- B2 字符级 tokenizer 对古诗友好,对现代中文(包含表情符号 / 罕见字)友好吗?
- B3 我们的 vocab 里有换行符 `\n` 吗?换行符的 ID 是几?它在模型里起什么作用?

**C. get_batch**
- C1 `block_size = 128` 意味着模型最多能看几个字的上文?如果上文超过 128 会怎样?
- C2 `batch_size = 32` 是什么意思?改成 64 / 128 训练会怎么变?
- C3 `torch.randint(low=0, high=N, size=(B,))` 抽出来可能有重复,这对训练有问题吗?

**D. 切分**
- D1 为什么不能把所有诗打乱再 90/10 切?和 Week 5(`prepare_v2.py`)里 `random.shuffle(poems)` 又有什么不同?
- D2 验证集只有训练集的 1/9,够吗?如果 val 只剩 1000 token 会怎样?

## 6. 容易踩的坑

**坑 1:`tensor view` 让 .pt 文件翻倍**

```python
data = torch.tensor(...)         # 假设 64 MB
train = data[:int(0.9*len(data))]   # 是个 view,共享底层 storage
val   = data[int(0.9*len(data)):]   # 也是 view,共享
torch.save(train, "train.pt")    # ← 实际写入了整个 64 MB storage,不是 58 MB!
torch.save(val, "val.pt")        # ← 也写入了整个 64 MB!
```

**修复**:存之前 `.clone()` 一下,切断 view 共享。注意 `data[:n]` / `data[n:]` 切出来都是 view —— `tokenizer/build.py` 和 Week 5 的 `prepare_v2.py` 现在都显式 `.clone()` 后再 `torch.save`。**不加 `.clone()` 会怎样**:`val.pt` 会和 `train.pt` 一样大(~32MB 而非 ~4MB),因为 `torch.save` 对 view 会写入整份底层 storage。

**坑 2:Windows GBK 控制台 + emoji 直接崩**

所有 `print("✅ 完成")` 在默认 cmd / PowerShell 里会 `UnicodeEncodeError: gbk codec`。
修复:不用 emoji,改成 `[OK]`;或者运行前 `chcp 65001` 切 UTF-8;或者 `$env:PYTHONIOENCODING="utf-8"`。

**坑 3:`stoi[c]` 遇到训练集没出现过的字符 → KeyError**

生成时如果 prompt 含训练集没见过的字(比如简体字 → 繁体训练),会崩。
应对:加 try/except 或加 `<|unk|>` token。Week 6 引入特殊 token 时会顺便讨论。

**坑 4:`paragraphs` 字段在某些 JSON 里是空 list**

部分残卷的 JSON 没有 `paragraphs` 字段(只有 `title`)。我们的 `data/merge.py` 用了 `poem.get("paragraphs", [])`,空 list 自动跳过 —— 但你要清楚这是有意的过滤,不是 bug。

## 7. 进入 Week 2 前

现在你应该:
- ☑ 跑完了 `data/prepare.py`,`data/processed/` 下有 4 个文件
- ☑ 能用 `tokenizer.encode("月")` 拿到一个数字
- ☑ 能解释 "为什么 `randint(len(d) - block_size - 1)` 是这个减法"

满足以上 3 条就开 Week 2。任何一条还模糊的,**别往下走**,回到对应小节再读一遍 + 跑一遍代码。

Week 2 我们正式上 Bigram → 单头 Attention → 多头 + FFN + 残差 → 6 层 Transformer,看 val loss 从 5.43 一路降到 4.14。